# 🕵️‍♂️ Analisis Silang Tebakan Test Set (LoRA vs Last Layer vs kNN)
Notebook ini akan menebak seluruh gambar di Test Set menggunakan checkpoint LoRA dan LastLayer (yang tadi baru dilatih 1 fold), lalu membandingkannya langsung dengan hasil tebakan dari kNN 5-fold milikmu.

Kita bisa menghitung **berapa banyak tebakannya yang BERBEDA (selisih)**, dan mencetak ID gambar apa saja yang menjadi sumber perdebatan antara model kNN dengan model LoRA.

In [ ]:
import os, sys, glob
import torch
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Pastikan repo di sys.path
REPO_DIR = '/content/satria-data-bdcugm02'
sys.path.insert(0, os.path.join(REPO_DIR, 'track_a', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'experiments'))

import config, lora_ft
from config import CFG
from transformers import AutoImageProcessor, AutoModel
from dataset import WasteDataset
from transforms import build_transforms
from torch.utils.data import DataLoader
from lora_ft import build_variant


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# 3. Load Kunci Jawaban kNN (Submission sebelumnya)
# Ganti dengan path file submission.csv hasil kNN-mu yang 16 salah itu!
KNN_SUBMISSION_PATH = os.path.join(CFG.save_dir, "..", "output_trackC", "submission_apace.csv")
if not os.path.exists(KNN_SUBMISSION_PATH):
    print(f"⚠️ {KNN_SUBMISSION_PATH} tidak ditemukan, pakai sample submission default")
    KNN_SUBMISSION_PATH = CFG.sample_sub_path

knn_df = pd.read_csv(KNN_SUBMISSION_PATH)
print(f"Loaded kNN submission: {len(knn_df)} rows")


In [ ]:
# 4. Load Processor & Siapkan Test Loader
CHECKPOINT = 'google/siglip2-so400m-patch14-384'
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
data_config = lora_ft.hf_processor_to_data_config(processor)

# Susun filepath urut sesuai dengan ID di submission.csv agar tidak tertukar posisinya!
test_images = [os.path.join(CFG.test_dir, str(img_id)) for img_id in knn_df['id']]
test_df = pd.DataFrame({'filepath': test_images, 'label': 0}) # 0 cuma dummy biar WasteDataset ga error
print(f"Ditemukan {len(test_df)} gambar test untuk di-inferensi.")

eval_tfm = build_transforms(data_config, CFG.img_size, train=False)
test_ds = WasteDataset(test_df, transform=eval_tfm)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
def predict_model(variant_name, run_name):
    print(f"\n--- 🔄 Mengeksekusi Inferensi {variant_name.upper()} ---")
    ckpt_path = os.path.join(CFG.save_dir, f"{run_name}_best.pt")
    if not os.path.exists(ckpt_path):
        print(f"❌ Checkpoint tidak ditemukan: {ckpt_path}")
        return None
        
    encoder = AutoModel.from_pretrained(CHECKPOINT).vision_model
    hidden_size = encoder.config.hidden_size
    model, _ = build_variant(variant_name, encoder, hidden_size, num_classes=3, n_last_blocks=4)
    
    print(f"✅ Loading weights dari {run_name}_best.pt...")
    state_dict = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    
    all_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(images)
            # Gunakan argmax (threshold 1.0) karena kita belum tuning OOF
            all_preds.append(outputs.argmax(dim=1).cpu().numpy())
            
    del model, encoder
    torch.cuda.empty_cache()
    return np.concatenate(all_preds)


In [ ]:
# 5. Lakukan Prediksi untuk LoRA dan Last Layer
preds_lora = predict_model('lora', 'lora_ft_fold0_5ep_v3')
if preds_lora is not None:
    knn_df['predicted_lora'] = preds_lora
    
preds_ll = predict_model('last_layer', 'last_layer_ft_fold0_5ep_v3')
if preds_ll is not None:
    knn_df['predicted_ll'] = preds_ll
    
knn_df.head()


In [ ]:
# 6. Hitung Disagreement (Selisih Tebakan)
print("==============================================")
print("📊 HASIL PERBANDINGAN DENGAN kNN")
print("==============================================")

if 'predicted_lora' in knn_df.columns:
    diff_lora = knn_df[knn_df['predicted'] != knn_df['predicted_lora']]
    print(f"\n🔥 Beda kNN vs LoRA: {len(diff_lora)} gambar ({len(diff_lora)/len(knn_df)*100:.1f}% dari total Test Set)")
    # Uncomment baris di bawah ini untuk melihat ID gambar mana saja yang beda:
    # display(diff_lora)
    
if 'predicted_ll' in knn_df.columns:
    diff_ll = knn_df[knn_df['predicted'] != knn_df['predicted_ll']]
    print(f"\n❄️ Beda kNN vs Last Layer: {len(diff_ll)} gambar ({len(diff_ll)/len(knn_df)*100:.1f}% dari total Test Set)")
    # display(diff_ll)

if 'predicted_lora' in knn_df.columns and 'predicted_ll' in knn_df.columns:
    diff_lora_ll = knn_df[knn_df['predicted_lora'] != knn_df['predicted_ll']]
    print(f"\n⚡ Beda LoRA vs Last Layer: {len(diff_lora_ll)} gambar ({len(diff_lora_ll)/len(knn_df)*100:.1f}% dari total Test Set)")
